In [ ]:
import sys
import subprocess

def install_dependencies():
    packages = ["mido", "pretty_midi", "tqdm", "torch", "numpy", "kagglehub[pandas-datasets]"]
    print("Checking dependencies...")
    for package in packages:
        try:
            __import__(package)
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print("Environment ready.")

install_dependencies()

In [ ]:
import torch

def average_models(model_paths, target_model):
    state_dicts = [torch.load(p)['model_state_dict'] for p in model_paths]
    new_state_dict = state_dicts[0]
    for key in new_state_dict:
        for i in range(1, len(state_dicts)):
            new_state_dict[key] += state_dicts[i][key]
        new_state_dict[key] /= len(state_dicts)
    target_model.load_state_dict(new_state_dict)

In [4]:
# Load from LOAD_PATH
import torch
from model4 import MusicTransformer

def load_model(path, device):
    checkpoint = torch.load(path, map_location=device)
    
    # Re-initialize the model with the saved settings
    model = MusicTransformer(
        vocab_size=391,
        d_model=512,
        max_len=1024
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("model_shard_0.pt", device=device)

In [7]:
# Top-k sampling
from tqdm.notebook import tqdm
import torch
import torch.nn.functional as F
import utils
tokenizer = utils.MIDITokenizer()


def generate_music(
    model,
    tokenizer,
    max_len=1024,
    top_k=10,
    temperature=0.5,
    device="cuda",
):
    model.eval()
    model = model.to(device)

    tokens = [tokenizer.sos_id]

    with torch.no_grad():
        for _ in tqdm(range(max_len), desc="Generating music"):
            x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

            logits = model(x)[:, -1, :] / temperature

            # Top-k filtering
            if top_k is not None:
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits[indices_to_remove] = float('-inf')
                
                probs = F.softmax(logits, dim=-1)
            else:
                probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, 1).item()

            if next_token == tokenizer.eos_id:
                break

            tokens.append(next_token)

    return tokens


generated_tokens = generate_music(
    model,
    tokenizer,
    device=device,
    max_len=100
)

print(f"Generated {len(generated_tokens)} tokens, sample: {generated_tokens[:50]}")
tokenizer.events_to_midi(generated_tokens, "out.mid")


Generating music:   0%|          | 0/100 [00:00<?, ?it/s]

Generated 101 tokens, sample: [389, 257, 257, 257, 257, 258, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 257, 258, 257, 257, 257, 258, 257, 257, 257, 257, 257, 257, 257, 257, 258, 257, 257, 258, 257, 257, 257]


'out.mid'

In [8]:
import pretty_midi
import numpy as np
from IPython.display import Audio, display

pm = pretty_midi.PrettyMIDI("out.mid")

audio = pm.synthesize(fs=44100)
audio = audio / np.max(np.abs(audio))

display(Audio(audio, rate=44100))


ValueError: zero-size array to reduction operation maximum which has no identity